# QAOA dynamics deep dive: one route, three search geometries

This notebook is the presentation-oriented companion to `src/`. It consumes the frozen files in `results/qaoa_dynamics_deep_dive/v1`; it does not rerun or retune optimization. Run it from either the repository root or `notebooks/`.

In [ ]:
from pathlib import Path
import csv, json
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'data' / 'graph.json').exists():
    ROOT = ROOT.parent
RESULTS = ROOT / 'results' / 'qaoa_dynamics_deep_dive' / 'v1'
FIGURES = ROOT / 'figures' / 'qaoa_dynamics_deep_dive' / 'v1'
summary = json.loads((RESULTS / 'final_summary.json').read_text())
validation = json.loads((RESULTS / 'scientific_validations.json').read_text())
energy_summary = json.loads((RESULTS / 'energy_landscape_summary.json').read_text())
with (RESULTS / 'dynamics_trace.csv').open(newline='') as handle:
    traces = list(csv.DictReader(handle))
len(summary), RESULTS

## 1. Fixed routing graph

The source is 0, the target is 6, and the unique shortest route is `0 -> 1 -> 2 -> 4 -> 5 -> 6` with cost 10.

In [ ]:
display(Image(filename=str(FIGURES / '01_fixed_weighted_routing_graph.png'), width=900))

## 2-5. Edge bits, exact reference, QUBO, and Ising $H_C$

One bit selects each of the 14 ordered directed edges. Full-space methods therefore use 16,384 basis states. The flow residual is $f_v=\sum_{out}x_e-\sum_{in}x_e-b_v$, and the frozen objective is $Q_6(x)=\sum_e w_ex_e+6\sum_v f_v(x)^2$. The mapping $x=(1-Z)/2$ gives the diagonal Ising Hamiltonian.

In [ ]:
print('full states:', energy_summary['state_count'])
print('feasible states:', energy_summary['feasible_state_count'])
print('unique optimum energy:', energy_summary['optimal_energy'])
print('QUBO/Ising max error:', validation['qubo_ising_max_error'])
print('Penalty-X / Global-Grover H_C byte-identical:', validation['penalty_x_global_hc_byte_identical'])

In [ ]:
display(Image(filename=str(FIGURES / '02_qubo_ising_qaoa_pipeline.png'), width=1000))

## 6-9. X, Global-Grover, and Feasible-Grover mixers

Penalty-X uses the local hypercube mixer $H_X=\sum_jX_j$. Global-Grover uses $H_{G,all}=|s_{all}\rangle\langle s_{all}|$ across all bitstrings. Feasible-Grover preserves the existing convention $H_{G,F}=|s_F\rangle\langle s_F|$ inside the enumerated route basis. Both projectors use $U_G=I+(e^{-i\beta}-1)P$.

In [ ]:
display(Image(filename=str(FIGURES / '03_three_search_spaces_and_mixers.png'), width=1000))
print('full initial p_opt =', 1 / 16384)
print('feasible initial p_opt =', 1 / 20)

## 10-12. One QAOA layer: cost phase, then mixer interference

The cost step maps $a_x$ to $a_xe^{-i\gamma E_x}$. It changes relative phase but not $|a_x|^2$, and it commutes with its own Hamiltonian. A following mixer can convert that phase structure into probability redistribution.

In [ ]:
with (RESULTS / 'physics_validation.csv').open(newline='') as handle:
    physics = list(csv.DictReader(handle))
print('max cost-step probability delta:', max(float(row['cost_probability_max_delta']) for row in physics))
print('max |cost-step energy delta|:', max(abs(float(row['cost_energy_delta'])) for row in physics))
print('any real mixer redistribution:', any(row['mixer_changed_probability'] == 'True' for row in physics))
display(Image(filename=str(FIGURES / '10_phase_and_interference_evolution.png'), width=1100))

## 13-14. Primary p=1 and p=2 results

The fixed policy is COBYLA, seed 2601, one deterministic initialization, and a 100-request cap. The p=1 grid is descriptive only and was not used to retune these runs.

In [ ]:
columns = ('algorithm', 'p', 'search_dimension', 'p_feas', 'p_opt', 'invalid_mass', 'expected_hc', 'optimizer_evaluations')
print(' | '.join(columns))
for row in summary:
    print(' | '.join(str(row[key]) for key in columns))

In [ ]:
display(Image(filename=str(FIGURES / '06_probability_mass_decomposition.png'), width=900))
display(Image(filename=str(FIGURES / '07_layer_by_layer_expected_hc.png'), width=900))

## 15. Layer-by-layer energy, feasibility, probability, and phase

For p=2 there are five checkpoints: Initial, Cost-1, Mixer-1, Cost-2, Mixer-2. Cost checkpoints repeat the preceding probability and energy values; mixer checkpoints are where redistribution can occur.

In [ ]:
for algorithm in ('penalty_x', 'grover_global', 'grover_feasible'):
    print('\n', algorithm)
    for row in traces:
        if row['algorithm'] == algorithm and row['p'] == '2':
            print(row['checkpoint'], 'E=', f"{float(row['expected_hc']):.6f}", 'p_feas=', f"{float(row['p_feas']):.8f}", 'p_opt=', f"{float(row['p_opt']):.8f}")

## 16. Search-space geometry

In [ ]:
display(Image(filename=str(FIGURES / '12_conceptual_search_space_geometry.png'), width=1000))

## 17. Interpretation

Penalty-X and Global-Grover isolate a mixer change under the same full representation and cost diagonal. Global-Grover and Feasible-Grover illustrate support restriction under analogous projector dynamics. The p=2 Grover runs redistribute probability, but an intermediate mixer can worsen the energy before a later mixer improves it.

In [ ]:
display(Image(filename=str(FIGURES / '11_energy_landscape.png'), width=1000))
for name in ('15_p1_parameter_landscape_penalty_x.png', '16_p1_parameter_landscape_grover_global.png', '17_p1_parameter_landscape_grover_feasible.png'):
    display(Image(filename=str(FIGURES / name), width=900))

## 18. Limitations

This is one 7-node graph, ideal simulation, one optimizer seed, and p<=2. Explicit feasible-route enumeration makes the optimum classically visible by `argmin` and is not scalable. A logical projector is not automatically hardware efficient. Structural `p_feas=1` is not a quantum-advantage result, and no general mixer ranking follows from these six cells.